In [ ]:
import torch
import numpy as np
from torch import nn
from torch.nn import functional as F
from torch import optim
from torch.utils.data import Dataset
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torchvision.utils import make_grid
from pathlib import Path
import os
from PIL import Image



In [ ]:
class IndexedDataset(Dataset):
    def __init__(self, index_file, transform=None):
        self.index_file = index_file
        self.transform = transform
        self.data = []
        self.labels = []
        self.load_data()

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        path = self.data[idx]
        label = self.labels[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label

    def load_data(self):
        with open(self.index_file, "r") as f:
            for line in f:
                path, label = line.strip().split()
                self.data.append(path)
                self.labels.append(int(label))

In [ ]:
emb_dim = 64

In [ ]:
data_dir = 'data'

transform = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

data_root = '~/Datasets/'

train_dataset = IndexedDataset(
    os.path.join(data_root / 'ms1m-arcface', 'index.txt')
)

test_dataset = IndexedDataset(
    os.path.join(data_root / 'LFW', 'index.txt')
)

train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=256,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

In [ ]:
class MiniCNN(nn.Module):
    def __init__(self):
        super(MiniCNN, self).__init__()
        

        self.encoder = nn.Sequential(
            # mix rgb channels
            nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            
            # depthwise 1
            nn.Conv2d(32, 32, kernel_size=3, stride=2, padding=1, groups=32),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            # pointwise 1
            nn.Conv2d(32, 64, kernel_size=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            # depthwise 2
            nn.Conv2d(64, 64, kernel_size=3, stride=2, padding=1, groups=64),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            # pointwise 2
            nn.Conv2d(64, 128, kernel_size=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU()            
            
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.LazyLinear(emb_dim)

    def forward(self, x):
        x = self.encoder(x)
        x = self.pool(x).flatten(1)
        x = self.fc(x)
        x = F.normalize(x, p=2, dim=1)
        return x
    

        